In [1]:
import sys
from pathlib import Path

root_path = Path.cwd().parent.parent

if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

In [2]:
# Truco para que Jupyter lea siempre la última versión de la carpeta src/
%load_ext autoreload
%autoreload 2

In [3]:
from src.data_loader import loader
from src.model_trainer import optimizar_hiperparametros_modelo
from src.evaluator import evaluar_modelo

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping

import joblib
import config

ModuleNotFoundError: No module named 'src'

In [ ]:
X_train, X_test, y_train, y_test = loader(OHE=True)

print("¡Datos cargados con éxito! Tamaño de entrenamiento:", X_train.shape)

#### RandomForestCalssifier
- se hace una búsqueda de hiperparámetros óptimos vía GridSearchCV
- se saca la importancia de las features en la clasificación
- se scan las métricas de clasificación
- se serializa el modelo en un pkl

In [336]:
modelo_rfc = RandomForestClassifier()

dict_parametros = {
    'n_estimators': [50, 100, 300],
    'max_depth': [8,10,12], #[6,8,10],
    'class_weight': ['balanced'],
    'n_jobs': [-1]
}
# Entrenamiento de un modelo de RandomForest Classifier
mejor_rfc, hiperparametros = optimizar_hiperparametros_modelo(modelo_rfc, dict_parametros, X_train, y_train)

print('¡Entrenamiento terminado! Los mejores parámetros son:\n', hiperparametros)

Mejores hiperparámetros encontrados: {'class_weight': 'balanced', 'max_depth': 12, 'n_estimators': 300, 'n_jobs': -1}
Mejor score obtenido: 80.97%


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [ ]:
# Métricas de evaluación
#evaluar_modelo(mejor_rfc, X_train, X_test, y_train, y_test, 'RandomForestClassifier')
y_pred_train = mejor_rfc.predict(X_train)
y_pred_test = mejor_rfc.predict(X_test)
y_proba_test = mejor_rfc.predict_proba(X_test)
evaluar_modelo(y_train, y_test, y_pred_train, y_pred_test, y_proba_test, 'RandomForestClassifier')

In [ ]:
# Guardar el modelo en un .pkl
config.MODELS_TEST_DIR.mkdir(parents=True, exist_ok=True)

# 2. Guardamos rfc ganador usando la ruta de tu config.py
joblib.dump(mejor_arbol, config.RFC_MODEL_PATH)

print(f"¡Modelo guardado con éxito en: {config.RFC_MODEL_PATH}!")

In [338]:
# Feature importance
importances = mejor_rfc.feature_importances_
feature_imp = pd.DataFrame({'Feature': X_train.columns, 'Gini Importance': importances}).sort_values('Gini Importance', ascending=False)
feature_imp.head(15)

,Feature,Gini Importance
47,deposit_type_Non Refund,0.138682
14,country_PRT,0.119329
415,lead_time,0.099751
426,total_of_special_requests,0.067021
425,required_car_parking_spaces,0.047185
420,previous_cancellations,0.045252
422,booking_changes,0.032204
413,customer_type_Transient,0.031962
18,market_segment_Groups,0.030139
20,market_segment_Online TA,0.024243


In [339]:
import re
feature_importance = {c:0 for c in dset.columns}
for index,row in feature_imp.iterrows():
    for feature in feature_importance.keys():
        sr = re.search(feature, row['Feature'])
        if sr:
            feature_importance[sr.group()] += row['Gini Importance']

In [340]:
pd.Series(dict(sorted(feature_importance.items(), key = lambda item: item[1], reverse=True)))

country                           0.175340
deposit_type                      0.138814
lead_time                         0.099751
market_segment                    0.087951
agent                             0.080158
total_of_special_requests         0.067021
customer_type                     0.056621
required_car_parking_spaces       0.047185
previous_cancellations            0.045252
distribution_channel              0.035783
booking_changes                   0.032204
adr                               0.023903
assigned_room_type                0.019520
company                           0.015531
hotel                             0.011393
stays_in_week_nights              0.011224
previous_bookings_not_canceled    0.010543
arrival_date_week_number          0.008435
reserved_room_type                0.007703
meal                              0.005469
stays_in_weekend_nights           0.005467
days_in_waiting_list              0.005253
is_repeated_guest                 0.004877
arrival_dat

#### Neural Network

In [382]:
# Definimos la arquitectura del modelo
model_nn = models.Sequential(layers=[
    layers.Input(shape=(X_train.shape[1],), name='i1'),
    layers.Dense(128, activation='relu', name='h1'),
    layers.Dense(64, activation='relu', name='h2'),
    layers.Dense(32, activation='relu', name='h3'),
    layers.Dense(1, activation='sigmoid', name='o1')
])

# Compilamos el modelo
optimizer_adam = Adam(learning_rate=0.001)
model_nn.compile(optimizer=optimizer_adam, loss='binary_crossentropy', metrics=['recall'])

Se observa que el modelo converge rápidamente; ha estado bien cortar temprano porqué ya no había gananccia el loss y val_loss tenía tendencia creciente (la ganancia en loss iba a ser por overfitting)

In [383]:
%%time
# Se entrena el modelo con un callback de early stopping. Se aplic en el val_loss porqué no se quiere overfitting
callback = EarlyStopping(monitor="val_loss",
                         min_delta=0.02,
                         patience=3,  # número de veces con una ganancia por debajo de la mínima seguidas antes de parar
                         verbose=0,
                         mode="auto",
                         baseline=None,
                         restore_best_weights=False,
                         start_from_epoch=0,
                        )
history = model_nn.fit(X_train, y_train, epochs=100, validation_split=0.2, verbose=1, callbacks=[callback])

Epoch 1/100
2388/2388 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 0.3515 - recall: 0.7162 - val_loss: 0.3234 - val_recall: 0.7710
Epoch 2/100
2388/2388 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.3170 - recall: 0.7569 - val_loss: 0.3177 - val_recall: 0.8136
Epoch 3/100
2388/2388 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 0.3064 - recall: 0.7609 - val_loss: 0.3108 - val_recall: 0.7875
Epoch 4/100
2388/2388 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.3004 - recall: 0.7707 - val_loss: 0.3009 - val_recall: 0.7632
Epoch 5/100
2388/2388 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.2937 - recall: 0.7734 - val_loss: 0.3018 - val_recall: 0.7977
Epoch 6/100
2388/2388 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 0.2895 - recall: 0.7798 - val_loss: 0.3025 - val_recall: 0.7840
Epoch 7/100
2388/2388 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - loss: 0.2851 - recall: 0.7846 - val_loss: 0.3015 - val_recall: 0.7853
CPU times: user 1min 35s, sys: 31.2 s, total: 2min 6s
Wall time: 1min 11s


In [ ]:
print('¡Entrenamiento terminado!)

In [ ]:
def nn_predict(model, X, threshold=0.5: float, pred_or_proba='pred') -> (pd.core.series.Series, pd.core.series.Series):
    '''
    Retorna (y_pred, y_pred_proba) o un diccionario con código de error 'code' y mensaje 'msg' 
    threshold debe estar entre 0 y 1, es el umbral de probabilidad a partir del cual se toma el resultado como '1'
    pred_or_proba dbe valer o 'pred' o 'proba'
    '''    
    if not (threshold >= 0 and threshold <= 0.5):
        return {'code':1,
                'msg': 'threshold debe estar entre 0 y 1, incluidos'}
    y_proba = model_nn.predict(X_test) 
    y_pred_proba = pd.Series(y_proba.flatten())
    if pred_or_proba == 'proba':     
        return y_pred_proba    
    elif pred_or_proba =0 'pred':
        return y_pred_proba.map(lambda x: 1 if x > threshold else 0)
    else:
        return {'code':2,
                'msg': 'pred_or_proba debe valer pred o proba'}

In [ ]:
# Métricas de evaluación
y_pred_train = mejor_rfc.nn_predict(model_nn, X_train)
y_pred_test = mejor_rfc.nn_predict(model_nn, X_test)
y_proba_test = mejor_rfc.nn_predict(model_nn, X_train, pred_or_proba='proba')
evaluar_modelo(y_train, y_test, y_pred_train, y_pred_test, y_proba_test, 'NeuralNetwork')

In [ ]:
# Guardar el modelo en un .pkl
config.MODELS_TEST_DIR.mkdir(parents=True, exist_ok=True)

# 2. Guardamos rfc ganador usando la ruta de tu config.py
joblib.dump(model_nn, config.NN_MODEL_PATH)

print(f"¡Modelo guardado con éxito en: {config.NN_MODEL_PATH}!")